# NB04 — Definición de los modelos

## Propósito

Definir las **dos cabezas de clasificación** que entrenamos en NB05:

1. **MLP** (PyTorch) — red neuronal pequeña con una capa oculta.
2. **LightGBM** — gradient boosting de árboles.

Las dos cabezas reciben los mismos vectores de features pre-extraídos en NB03 y producen una probabilidad de clase positiva. Comparándolas sobre las mismas features respondemos: **¿qué tipo de cabeza explota mejor las representaciones de AsymMirai?**

## ¿Por qué dos cabezas tan distintas?

Tienen sesgos inductivos opuestos, y eso enriquece la comparación:

- **MLP**: aprende combinaciones **no lineales** de todas las features simultáneamente, pero produce probabilidades mal calibradas con BCE+pos_weight (los logits están sesgados por el peso del desbalance).
- **GBM**: aprende **reglas de decisión** sobre features individuales (splits de árboles), produce probabilidades **bien calibradas** de forma natural, y trata mejor features irrelevantes (las ignora).

Si las dos llegan al mismo AUC, sabemos que el ranking aprendido no depende del modelo: depende de las features. Si difieren, la diferencia es interpretable.

## Decisiones de diseño del MLP

Una capa oculta es suficiente y se justifica por:

- **Tamaño del problema**: ~385 positivos a nivel estudio en training. Arquitecturas más profundas sobreajustarían sin remedio.
- **Función de la cabeza**: no reentrenar el backbone (lo congelamos), solo aprender una combinación adecuada de sus features para la tarea binaria BI-RADS.
- **BatchNorm**: estabiliza el entrenamiento cuando las features tienen escalas distintas. Aquí concatenamos GAP (rango pequeño, media positiva) con GMP (picos altos), así que las escalas son muy diferentes y BN ayuda.
- **Dropout 0.3**: regularización suave, suficiente para un MLP pequeño.

**Capa oculta escalada con la entrada**: `hidden = max(64, in_dim // 16)`. Para in_dim=4096 → 256, para in_dim=2048 → 128. Es una heurística clásica para mantener un ratio razonable de parámetros sin tener que ajustarlo a mano.

## Decisiones de diseño del GBM (LightGBM)

Hiperparámetros conservadores y comunes para tabular de tamaño medio:

- `n_estimators=300` con `early_stopping_rounds=30`. Si después de 30 iteraciones sin mejorar el AUC de validación, para.
- `learning_rate=0.05`: paso pequeño para evitar sobreajuste rápido.
- `num_leaves=31`: árboles relativamente simples (~5 niveles).
- `min_child_samples=20`: cada hoja debe tener al menos 20 muestras (evita splits espurios).
- `reg_alpha=0.1`, `reg_lambda=0.1`: regularización L1/L2 suave.

**LightGBM sobre XGBoost** por dos motivos prácticos: más rápido en CPU para nuestros tamaños, y manejo natural del desbalance con `scale_pos_weight`.

## Tratamiento del desbalance — el mismo en ambas cabezas

Las dos cabezas reciben el mismo factor `n_neg / n_pos`:

- MLP: `BCEWithLogitsLoss(pos_weight=n_neg/n_pos)` multiplica el gradiente de las muestras positivas por ese factor.
- LightGBM: `scale_pos_weight=n_neg/n_pos` reescala el peso de las muestras positivas en la pérdida.

Conceptualmente hacen lo mismo. Esto asegura que la comparación MLP vs GBM no se vea afectada por una gestión distinta del desbalance.

## Salida del notebook

Se exporta un módulo `tfm_models.py` con `build_mlp(in_dim)`, `build_gbm(scale_pos_weight)` y la clase `Head`. NB05 lo importa.

In [1]:
import os
import torch
import torch.nn as nn

# Raíz del proyecto: por defecto, la carpeta padre de notebooks/.
# Sobrescribible con la variable de entorno TFM_PROJECT_ROOT.
BASE      = os.environ.get('TFM_PROJECT_ROOT',
                           os.path.abspath(os.path.join(os.getcwd(), '..')))
NB_DIR    = os.path.join(BASE, 'src')
os.makedirs(NB_DIR, exist_ok=True)

try:
    import lightgbm as lgb
    print(f'LightGBM disponible: v{lgb.__version__}')
except ImportError:
    print('LightGBM NO disponible. Instala con: pip install lightgbm')

LightGBM disponible: v4.6.0


## 1. Definición del MLP

Cabeza adaptable a cualquier `in_dim`. La estructura es:

```
Linear(in_dim → hidden) → BatchNorm1d → ReLU → Dropout(0.3) → Linear(hidden → 1)
```

Devuelve **logits** (no probabilidades), porque `BCEWithLogitsLoss` es numéricamente más estable que aplicar `sigmoid` + `BCELoss` por separado.

In [2]:
class Head(nn.Module):
    """
    MLP sencilla con BatchNorm y Dropout, salida en logits (sin sigmoide).

    Args:
        in_dim: dimensión del vector de entrada.
        hidden: dimensión de la capa oculta. Si None, se calcula como max(64, in_dim // 16).
        dropout: probabilidad de dropout en la capa oculta.
    """
    def __init__(self, in_dim, hidden=None, dropout=0.3):
        super().__init__()
        if hidden is None:
            hidden = max(64, in_dim // 16)
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(hidden, 1),
        )
        self.in_dim, self.hidden = in_dim, hidden
    def forward(self, x):
        return self.net(x)
    def __repr__(self):
        return f'Head({self.in_dim} → {self.hidden} → 1)'

def build_mlp(in_dim):
    return Head(in_dim=in_dim, dropout=0.3)

# Sanity check para las dimensionalidades que usaremos
for in_dim in [4096, 2048]:
    m = build_mlp(in_dim)
    n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    x = torch.randn(8, in_dim)
    print(f'  in_dim={in_dim:5d}  →  {m}   params={n_params:,}   forward({tuple(x.shape)}) → {tuple(m(x).shape)}')

  in_dim= 4096  →  Head(4096 → 256 → 1)   params=1,049,601   forward((8, 4096)) → (8, 1)
  in_dim= 2048  →  Head(2048 → 128 → 1)   params=262,657   forward((8, 2048)) → (8, 1)


## 2. Builder de LightGBM

Cada vez que se llama, devuelve un clasificador nuevo con los hiperparámetros fijados y el `scale_pos_weight` que indiquemos (lo ajustaremos por fold en NB05).

In [3]:
import lightgbm as lgb

def build_gbm(scale_pos_weight=1.0, seed=42):
    """
    Crea un clasificador LightGBM con hiperparámetros conservadores.
    Args:
        scale_pos_weight: peso relativo de la clase positiva (típicamente n_neg/n_pos del fold).
        seed: para reproducibilidad.
    """
    return lgb.LGBMClassifier(
        n_estimators       = 300,
        learning_rate      = 0.05,
        num_leaves         = 31,
        min_child_samples  = 20,
        reg_alpha          = 0.1,
        reg_lambda         = 0.1,
        scale_pos_weight   = scale_pos_weight,
        random_state       = seed,
        n_jobs             = -1,
        verbosity          = -1,
    )

gbm = build_gbm(scale_pos_weight=10.0)
print(f'  GBM: {type(gbm).__name__}')
print(f'  scale_pos_weight = {gbm.scale_pos_weight}')
print(f'  n_estimators={gbm.n_estimators}, lr={gbm.learning_rate}, num_leaves={gbm.num_leaves}')

  GBM: LGBMClassifier
  scale_pos_weight = 10.0
  n_estimators=300, lr=0.05, num_leaves=31


## 3. Exportar como módulo importable

Escribimos `tfm_models.py` en la misma carpeta de notebooks. Así NB05, NB07 y cualquier notebook futuro pueden hacer `from tfm_models import ...` sin redefinir nada.

**Nota sobre Pylance:** VS Code puede mostrar el warning `Import "tfm_models" could not be resolved`. Es solo del análisis estático del editor; **en runtime el import funciona** porque NB05 hace `sys.path.insert(0, NB_DIR)` antes de importar.

In [4]:
MODULE_CODE = '''"""
tfm_models — cabezas de clasificación del TFM.
Uso:
    from tfm_models import Head, build_mlp, build_gbm
"""
import torch.nn as nn
import lightgbm as lgb

class Head(nn.Module):
    def __init__(self, in_dim, hidden=None, dropout=0.3):
        super().__init__()
        if hidden is None:
            hidden = max(64, in_dim // 16)
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(hidden, 1),
        )
        self.in_dim, self.hidden = in_dim, hidden
    def forward(self, x):
        return self.net(x)

def build_mlp(in_dim):
    return Head(in_dim=in_dim, dropout=0.3)

def build_gbm(scale_pos_weight=1.0, seed=42):
    return lgb.LGBMClassifier(
        n_estimators=300, learning_rate=0.05, num_leaves=31,
        min_child_samples=20, reg_alpha=0.1, reg_lambda=0.1,
        scale_pos_weight=scale_pos_weight, random_state=seed,
        n_jobs=-1, verbosity=-1,
    )
'''

out_path = os.path.join(NB_DIR, 'tfm_models.py')
with open(out_path, 'w', encoding='utf-8') as f:
    f.write(MODULE_CODE)
print(f'Módulo escrito: {out_path}')

Módulo escrito: C:\Users\victo\Documents\TFM\Proyecto\Notebooks\tfm_models.py


## Siguiente paso

`05_entrenamiento_kfold.ipynb` — StratifiedKFold(5) sobre `split=training`, entrenando las 6 configuraciones (E-A, E-B, M-A × MLP, GBM) y guardando predicciones OOF y ensemble sobre test.